# Resultados, inferencia estadística y diseño de estrategias
**Optimización del portafolio de variedades y la asignación de superficie para maximizar el margen por metro cuadrado: el caso de una florícola exportadora de rosas**

Bryan Steve Vargas Cabascango — Maestría en Inteligencia de Negocios y Ciencia de Datos

---

Tercer cuaderno del proyecto. Requiere haber ejecutado antes, en este orden:

1. `01_EDA_Portafolio_Rosas.ipynb` → genera `figuras/panel_analitico.pkl`
2. `02_Modelado_Predictivo_Prescriptivo.ipynb` → genera `modelo/residuos.pkl` y `modelo/asignacion.csv`

Este cuaderno añade la **inferencia estadística** sobre los resultados:

| Sección | Pregunta que responde |
|---|---|
| 2 | ¿La mejora del modelo predictivo es estadísticamente significativa? (Diebold-Mariano) |
| 3 | ¿Qué variedades tienen un margen significativamente distinto de cero? (Newey-West) |
| 4 | ¿Qué tan seguro es el costo de oportunidad? (bootstrap por bloques) |
| 5 | ¿Qué decisiones de la cartera son estables y cuáles frágiles? |
| 6 | ¿Cuánto se gana ejecutando solo las decisiones robustas? |
| 7 | ¿Es económicamente viable la renovación? (VAN y umbral de costo) |

## 1. Configuración

In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from sklearn.covariance import LedoitWolf

PANEL = './figuras/panel_analitico.pkl'
RESID = './modelo/residuos.pkl'
ASIG  = './modelo/asignacion.csv'
SALIDA = './resultados/'
os.makedirs(SALIDA, exist_ok=True)

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.grid':True,
                     'grid.alpha':.3,'figure.dpi':110,
                     'axes.spines.top':False,'axes.spines.right':False})
AZ, RO, VE, MO, GR, NA = '#4C72B0','#C44E52','#55A868','#8172B2','#BBBBBB','#DD8452'

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(f'{SALIDA}{nombre}.png', dpi=150, bbox_inches='tight')
    plt.show()

def nw_se(x, L):
    """Error estándar de la media con corrección de Newey-West (rezago L)."""
    x = np.asarray(x, float); x = x[~np.isnan(x)]; T = len(x)
    e = x - x.mean(); s = e @ e / T
    for l in range(1, min(L, T-1) + 1):
        s += 2 * (1 - l/(L+1)) * (e[l:] @ e[:-l] / T)
    return np.sqrt(max(s, 1e-18) / T), T

d = pd.read_pickle(PANEL).sort_values(['variety','id_month'])
R = pd.read_pickle(RESID)
A = pd.read_csv(ASIG)
inferencia = {}
print(f'Panel: {len(d):,} obs | Residuos: {len(R):,} obs | Asignación: {len(A)} variedades')

## 2. ¿La mejora del modelo es significativa? Prueba de Diebold-Mariano

Compara el error cuadrático de Random Forest con el de la media histórica en el
período de prueba. El diferencial de pérdida se agrega por mes para evitar la
dependencia entre variedades de un mismo mes, y se aplica la corrección de
Harvey, Leybourne y Newbold para muestras pequeñas.

$$d_t = \overline{(m - m^{\text{hist}})^2}_t - \overline{(m - \hat{m}^{\text{RF}})^2}_t
\qquad H_0:\; E[d_t] = 0 \quad H_1:\; E[d_t] > 0$$

In [ ]:
g = d.groupby('variety', group_keys=False)
for L in (1,2,3,12): d[f'lag{L}'] = g.margin_m2.shift(L)
d['roll3']    = g.margin_m2.transform(lambda s: s.shift(1).rolling(3).mean())
d['roll12']   = g.margin_m2.transform(lambda s: s.shift(1).rolling(12).mean())
d['exp_mean'] = g.margin_m2.transform(lambda s: s.shift(1).expanding().mean())
d['exp_std']  = g.margin_m2.transform(lambda s: s.shift(1).expanding().std())
d['lag_stems']=g.stems_m2.shift(1); d['lag_nac']=g.pct_nacional.shift(1)
d['lag_nocol']=g.pct_no_colocada.shift(1); d['lag_price']=g.avg_price.shift(1)
d['mes']=d.id_month.dt.month; d['trend']=(d.id_month.dt.year-2020)*12+d.mes
FEATS=['lag1','lag2','lag3','lag12','roll3','roll12','exp_mean','exp_std','lag_stems',
       'lag_nac','lag_nocol','lag_price','mes','trend','density','monthly_flowers_per_plant',
       'area','n_plants','is_valentin','is_mothers_day_season','is_all_saints_season',
       'is_christmas_season','is_covid_shock']
dm = d.dropna(subset=['lag12','roll12','exp_mean','margin_m2']).copy()
dm['exp_std'] = dm.exp_std.fillna(0)

tr, te = dm[dm.id_month < '2025-01-01'], dm[dm.id_month >= '2025-01-01'].copy()
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=5, max_features=.5,
                           random_state=42, n_jobs=-1).fit(tr[FEATS], tr.margin_m2)
te['p_rf'] = rf.predict(te[FEATS])
te['dl'] = (te.margin_m2 - te.exp_mean)**2 - (te.margin_m2 - te.p_rf)**2

dmes = te.groupby('id_month').dl.mean()
se, T = nw_se(dmes.values, 2)
dm_stat = dmes.mean() / se
h = 2
hln = dm_stat * np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
p_dm = 1 - stats.t.cdf(hln, df=T-1)
gana = int((dmes > 0).sum())
p_signo = 1 - stats.binom.cdf(gana-1, T, .5)

inferencia['diebold_mariano'] = dict(meses=int(T), dif_media=float(dmes.mean()),
    estadistico_hln=float(hln), p_valor=float(p_dm), meses_rf_gana=gana, p_signo=float(p_signo))
print(f'Diferencial medio de pérdida: {dmes.mean():.4f}')
print(f'Estadístico DM (HLN): {hln:.3f}   p-valor unilateral: {p_dm:.4f}')
print(f'Random Forest gana en {gana} de {T} meses (prueba de signo p = {p_signo:.3f})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dmes.index, dmes.values, width=20, color=np.where(dmes.values > 0, VE, RO))
ax.axhline(0, color='k', lw=.9)
ax.axhline(dmes.mean(), color=AZ, ls='--', lw=1.4, label=f'Media = {dmes.mean():.3f}  (p = {p_dm:.3f})')
ax.set_ylabel('Reducción del error cuadrático')
ax.set_title('Ventaja mensual de Random Forest sobre la media histórica')
ax.legend()
guardar(fig, 'r01_diebold_mariano')

## 3. ¿Qué variedades tienen un margen significativamente distinto de cero?

Prueba t sobre el margen medio mensual de cada variedad, con error estándar de
Newey-West (rezago 3) para corregir la autocorrelación del margen crudo.

In [ ]:
rows = []
for v, gv in d.groupby('variety'):
    x = gv.sort_values('id_month').margin_m2.values
    se_v, Tv = nw_se(x, 3)
    tstat = x.mean() / se_v
    rows.append(dict(variety=v, meses=Tv, anual=x.mean()*12, t=tstat,
                     p=2*(1 - stats.t.cdf(abs(tstat), Tv-1)), area=gv.area.mean()))
V = pd.DataFrame(rows)
V['clase'] = np.select([V.t < -1.96, V.t > 1.96],
                       ['Negativo significativo','Positivo significativo'], 'No distinguible de cero')
V['share'] = V.area / V.area.sum()
V.to_csv(SALIDA + 'significancia_variedades.csv', index=False)

cl = V.groupby('clase').agg(n=('variety','count'), sup=('share','sum'))
inferencia['significancia'] = {k: dict(n=int(r.n), superficie_pct=float(r.sup*100)) for k, r in cl.iterrows()}
(cl.assign(sup=cl.sup*100).rename(columns={'n':'Variedades','sup':'Superficie (%)'}).round(1))

In [ ]:
orden = ['Negativo significativo','No distinguible de cero','Positivo significativo']
fig, ax = plt.subplots(figsize=(7.5, 4.2))
b = ax.bar(orden, [cl.loc[o,'sup']*100 for o in orden], color=[RO, GR, VE], width=.55)
for r_, o in zip(b, orden):
    ax.text(r_.get_x()+r_.get_width()/2, r_.get_height(),
            f'{int(cl.loc[o,"n"])} variedades\n{cl.loc[o,"sup"]*100:.1f}%', ha='center', va='bottom', fontsize=9.5)
ax.set_ylabel('Participación en la superficie (%)'); ax.set_ylim(0, cl.sup.max()*100*1.25)
ax.set_title('Clasificación de variedades por significancia del margen (α = 0,05)')
guardar(fig, 'r02_significancia_variedades')

## 4. ¿Qué tan seguro es el costo de oportunidad? Bootstrap por bloques

Se remuestrean bloques de 3 meses consecutivos de la matriz de residuos y de la
matriz de márgenes observados. En cada réplica se reestima la matriz de
Ledoit-Wolf, el margen esperado (media histórica) y se resuelve de nuevo la
optimización. Los bloques preservan la dependencia temporal de corto plazo.

Se usan **gradientes analíticos** en el optimizador: cada réplica tarda ~0,2 s.

In [ ]:
Rw = R[R.id_month >= '2022-01-01']
M = Rw.pivot_table(index='id_month', columns='variety', values='resid')
M = M.loc[:, M.notna().mean() >= .95].fillna(0.); VAR = list(M.columns); n = len(VAR)
Hh = d[d.id_month >= '2022-01-01'].pivot_table(index='id_month', columns='variety', values='margin_m2')[VAR]
Hh = Hh.fillna(Hh.mean()).loc[M.index]

ultimo = d.id_month.max()
ar = d[d.id_month == ultimo].set_index('variety').area.reindex(VAR).fillna(0)
w0 = (ar/ar.sum()).values; AT = float(ar.sum())
CAP, CRECE, TUR = .25, 2., .20
UB = np.minimum(CAP, np.maximum(w0*CRECE, .005))
I_ = np.eye(n)

def resolver(mu, S, s_obj, lb=None, ub=None, tur=TUR):
    """Máximo margen a un riesgo dado, con rotación L1 reformulada (gradientes analíticos).

    SLSQP puede no converger con tolerancia estricta según la versión de SciPy, por lo que
    se reintenta con tolerancias más laxas y un segundo punto de partida. Devuelve None solo
    si ningún intento converge.
    """
    lb = np.zeros(n) if lb is None else lb
    ub = UB if ub is None else ub
    cons = [{'type':'eq','fun':lambda z: z[:n].sum()-1,'jac':lambda z: np.r_[np.ones(n),np.zeros(n)]},
            {'type':'ineq','fun':lambda z: z[n:]-(z[:n]-w0),'jac':lambda z: np.hstack([-I_,I_])},
            {'type':'ineq','fun':lambda z: z[n:]+(z[:n]-w0),'jac':lambda z: np.hstack([I_,I_])},
            {'type':'ineq','fun':lambda z: 2*tur-z[n:].sum(),'jac':lambda z: np.r_[np.zeros(n),-np.ones(n)]},
            {'type':'ineq','fun':lambda z: s_obj**2-z[:n]@S@z[:n],'jac':lambda z: np.r_[-2*S@z[:n],np.zeros(n)]}]
    bnds = list(zip(lb, ub)) + [(0,1)]*n
    medio = np.clip((w0 + ub[:n])/2, 0, None); medio = medio/medio.sum()
    for ftol in (1e-10, 1e-8, 1e-7):
        for z0 in (np.r_[w0, np.zeros(n)], np.r_[medio, np.zeros(n)]):
            r = minimize(lambda z: -(mu@z[:n]), z0, jac=lambda z: np.r_[-mu,np.zeros(n)],
                         method='SLSQP', bounds=bnds, constraints=cons,
                         options={'maxiter':2000,'ftol':ftol})
            if r.success:
                return r.x[:n]
    return None

S0 = LedoitWolf().fit(M.values).covariance_ * 12
mu0 = Hh.mean().values * 12
w_pt = resolver(mu0, S0, np.sqrt(w0@S0@w0))
assert w_pt is not None, 'El optimizador no convergió; revise versiones de SciPy'
gan_pt = float(mu0@w_pt - mu0@w0)
print(f'Estimación puntual (media histórica): {gan_pt:.3f} USD/m²/año')

In [ ]:
rng = np.random.default_rng(42)
B, BL = 500, 3
Tm = len(M); nb = int(np.ceil(Tm/BL))
gan, elim, amp = [], np.zeros(n), np.zeros(n)
for _ in range(B):
    ini = rng.integers(0, Tm-BL+1, nb)
    idx = np.concatenate([np.arange(s, s+BL) for s in ini])[:Tm]
    Sb = LedoitWolf().fit(M.values[idx]).covariance_ * 12
    mub = Hh.values[idx].mean(0) * 12
    wb = resolver(mub, Sb, np.sqrt(w0@Sb@w0))
    if wb is None:
        continue
    gan.append(float(mub@wb - mub@w0))
    elim += (w0 > 1e-4) & (wb <= 1e-4)
    amp += wb > w0 + 1e-4
gan = np.array(gan); Bv = len(gan); elim /= Bv; amp /= Bv
lo, hi = np.percentile(gan, [2.5, 97.5])

inferencia['bootstrap'] = dict(replicas=Bv, ganancia_puntual=gan_pt, media=float(gan.mean()),
    ic95=[float(lo), float(hi)], ic95_usd=[float(lo*AT), float(hi*AT)],
    prob_ganancia_positiva=float((gan > 0).mean()))
print(f'Réplicas válidas: {Bv} de {B}')
print(f'IC 95 %: [{lo:.3f}; {hi:.3f}] USD/m²/año  =  [{lo*AT:,.0f}; {hi*AT:,.0f}] USD/año')
print(f'Probabilidad de ganancia positiva: {(gan > 0).mean():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.hist(gan, bins=40, color=AZ, edgecolor='white', lw=.4)
ax.axvline(gan_pt, color='k', lw=1.6, label=f'Estimación puntual = {gan_pt:.2f}')
ax.axvline(lo, color=RO, ls='--', lw=1.4, label=f'IC 95 % = [{lo:.2f}; {hi:.2f}]')
ax.axvline(hi, color=RO, ls='--', lw=1.4)
ax.set_xlabel('Costo de oportunidad (USD por m² al año)'); ax.set_ylabel('Réplicas')
ax.set_title(f'Distribución bootstrap del costo de oportunidad ({Bv} réplicas)')
ax.legend(fontsize=9)
guardar(fig, 'r03_bootstrap_ganancia')

## 5. ¿Qué decisiones son estables?

Frecuencia con que cada variedad es eliminada o ampliada a lo largo de las réplicas.
Una decisión se considera **robusta** si la solución puntual la recomienda y el
bootstrap la confirma en al menos el 70 % de las réplicas.

In [ ]:
st = pd.DataFrame({'variety': VAR, 'p_eliminar': elim, 'p_ampliar': amp})
A2 = A.merge(st, on='variety').merge(V[['variety','t','clase']], on='variety', how='left')
A2.to_csv(SALIDA + 'asignacion_con_estabilidad.csv', index=False)

el = A2[A2.share_optimo < 1e-4].sort_values('p_eliminar', ascending=False).copy()
el['fase'] = np.select([el.p_eliminar >= .9, el.p_eliminar >= .7],
                       ['Fase 1 (T1-T2)','Fase 2 (T3)'], 'Revisión comercial')
el[['variety','area_actual_m2','margen_esperado_m2_anio','p_eliminar','clase','fase']].round(2)

In [ ]:
t = el.sort_values('p_eliminar')
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.barh(t.variety, t.p_eliminar*100,
        color=[RO if p >= .9 else (NA if p >= .7 else GR) for p in t.p_eliminar])
ax.axvline(90, color=RO, ls=':', lw=1.2); ax.axvline(70, color=NA, ls=':', lw=1.2)
ax.set_xlabel('Frecuencia de eliminación en el bootstrap (%)'); ax.set_xlim(0, 100)
ax.set_title('Estabilidad de la decisión de eliminar cada variedad')
guardar(fig, 'r04_estabilidad_eliminacion')

In [ ]:
# Ampliaciones: robustas vs fragiles
am = A2[A2.delta_m2 > 50].sort_values('delta_m2', ascending=False)
am[['variety','area_actual_m2','delta_m2','margen_esperado_m2_anio','p_ampliar']].round(2)

## 6. Escenarios de ejecución

| Escenario | Qué se permite |
|---|---|
| **Plan completo** | La solución puntual del optimizador |
| **Núcleo robusto** | Solo se reducen las variedades con eliminación robusta y solo se amplían las de ampliación robusta; el resto queda fijo |

Ambos se resuelven con el margen esperado del modelo predictivo y al mismo riesgo
que la cartera vigente.

In [ ]:
P = Rw.pivot_table(index='id_month', columns='variety', values='pred')[VAR]
mu = P.tail(12).mean().values * 12
S  = S0
s0 = float(np.sqrt(w0@S@w0)); r0 = float(mu@w0)
Ai = A2.set_index('variety').reindex(VAR)

# plan completo
wF = resolver(mu, S, s0)

# nucleo robusto
valid  = (Ai.share_optimo.values < 1e-4) & (Ai.p_eliminar.values >= .7)
amp_ok = (Ai.delta_m2.values > 50) & (Ai.p_ampliar.values >= .5)
lb = w0.copy(); lb[valid] = 0.0
ub = w0.copy(); ub[amp_ok] = UB[amp_ok]
cons = [{'type':'eq','fun':lambda z: z.sum()-1,'jac':lambda z: np.ones(n)},
        {'type':'ineq','fun':lambda z: s0**2-z@S@z,'jac':lambda z: -2*S@z}]
r = minimize(lambda z: -(mu@z), w0, jac=lambda z: -mu, method='SLSQP',
             bounds=list(zip(lb,ub)), constraints=cons, options={'maxiter':800,'ftol':1e-10})
wN = r.x

def resumen(w):
    rot = np.abs(w-w0).sum()/2
    return dict(margen=float(mu@w), ganancia_m2=float(mu@w-r0), ganancia_usd=float((mu@w-r0)*AT),
                rotacion_pct=float(rot*100), rotacion_m2=float(rot*AT),
                eliminadas=int(((w0>1e-4)&(w<=1e-4)).sum()),
                ganancia_por_m2_rotado=float((mu@w-r0)/rot))
# Escenario intermedio: se bloquean las decisiones frágiles, pero se permite que el
# optimizador compense reduciendo otras variedades no validadas (sustitución).
lb_s = np.zeros(n); ub_s = UB.copy()
frag_elim = (Ai.share_optimo.values < 1e-4) & (Ai.p_eliminar.values < .7)
frag_amp  = (Ai.delta_m2.values > 50) & (Ai.p_ampliar.values < .5)
lb_s[frag_elim] = w0[frag_elim]; ub_s[frag_elim] = w0[frag_elim]   # se mantienen fijas
ub_s[frag_amp] = w0[frag_amp]                                      # no se amplían
wS = resolver(mu, S, s0, lb=lb_s, ub=ub_s)
assert wS is not None, 'El optimizador no convergió en el escenario con sustitución'

esc = {'Plan completo': resumen(wF), 'Plan robusto con sustitución': resumen(wS),
       'Núcleo robusto': resumen(wN)}
esc['Núcleo robusto']['captura_pct'] = esc['Núcleo robusto']['ganancia_m2']/esc['Plan completo']['ganancia_m2']*100
inferencia['escenarios'] = esc
print(f'Variedades fijadas por eliminación frágil: {frag_elim.sum()} | '
      f'ampliaciones bloqueadas: {frag_amp.sum()}')
print(f'El escenario con sustitución elimina {int(((w0>1e-4)&(wS<=1e-4)).sum())} variedades, '
      f'frente a {int(((w0>1e-4)&(wN<=1e-4)).sum())} del núcleo robusto.')
pd.DataFrame(esc).T.round(2)

In [ ]:
orden_esc = ['Núcleo robusto', 'Plan robusto con sustitución', 'Plan completo']
etiq = ['Núcleo robusto', 'Plan robusto\ncon sustitución', 'Plan completo']
gan = [esc[k]['ganancia_usd']/1000 for k in orden_esc]
rot = [esc[k]['rotacion_pct'] for k in orden_esc]

fig, ax = plt.subplots(figsize=(7.5, 4.3))
b = ax.bar(etiq, gan, color=[VE, GR, AZ], width=.55)
for r_, g, rt in zip(b, gan, rot):
    ax.text(r_.get_x()+r_.get_width()/2, g, f'{g:,.0f} mil USD\nrotación {rt:.1f} %'.replace(',', '.'),
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Ganancia anual (miles de USD)')
ax.set_ylim(0, max(gan)*1.25)
ax.set_title('Ganancia anual por escenario de ejecución, a igual riesgo')
guardar(fig, 'r06_escenarios')

El escenario intermedio se descarta: al bloquear las decisiones frágiles, el optimizador
compensa eliminando otras variedades que **no fueron validadas por el bootstrap**, de modo
que sustituye un riesgo de decisión por otro equivalente. El núcleo robusto, en cambio,
solo ejecuta decisiones confirmadas.

In [ ]:
N = pd.DataFrame({'variety':VAR,'area_actual_m2':w0*AT,'area_nucleo_m2':wN*AT,'margen_esperado':mu})
N['delta_m2'] = N.area_nucleo_m2 - N.area_actual_m2
N.sort_values('delta_m2', ascending=False).to_csv(SALIDA + 'plan_nucleo_robusto.csv', index=False)

t = N[N.delta_m2.abs() > 50].sort_values('delta_m2')
fig, ax = plt.subplots(figsize=(7.5, 6.2))
ax.barh(t.variety, t.delta_m2, color=np.where(t.delta_m2 > 0, VE, RO))
ax.axvline(0, color='k', lw=.9)
ax.set_xlabel('Cambio de superficie (m²)')
ax.set_title('Plan de ejecución inmediata: núcleo robusto de decisiones')
guardar(fig, 'r07_nucleo_robusto')

## 7. Viabilidad económica

Valor actual neto por metro cuadrado rotado, con tasa de descuento del 12 %,
vida productiva de la cama de 7 años y **primer año a media producción** por el
período de maduración de la planta.

$$\text{VAN} = g \left[\frac{0{,}5}{1+r} + \sum_{k=2}^{7} \frac{1}{(1+r)^k}\right] - C$$

donde $g$ es la ganancia anual por m² rotado y $C$ el costo de renovación por m².

In [ ]:
TASA, VIDA = .12, 7
factor = .5/(1+TASA) + sum(1/(1+TASA)**k for k in range(2, VIDA+1))
gF = esc['Plan completo']['ganancia_por_m2_rotado']
gN = esc['Núcleo robusto']['ganancia_por_m2_rotado']
inferencia['viabilidad'] = dict(factor=factor, umbral_completo=gF*factor, umbral_nucleo=gN*factor,
                                payback_nucleo_costo_40=40/gN + .5)
print(f'Factor de anualidad con maduración: {factor:.3f}')
print(f'Umbral de costo de renovación — plan completo:  {gF*factor:.1f} USD/m²')
print(f'Umbral de costo de renovación — núcleo robusto: {gN*factor:.1f} USD/m²')

c_ = np.arange(0, 161, 5)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(c_, gN*factor - c_, color=VE, lw=2.2, label=f'Núcleo robusto (umbral {gN*factor:.0f} USD/m²)')
ax.plot(c_, gF*factor - c_, color=AZ, lw=2, ls='--', label=f'Plan completo (umbral {gF*factor:.0f} USD/m²)')
ax.axhline(0, color='k', lw=.9)
ax.set_xlabel('Costo de renovación (USD por m², planta y labores)')
ax.set_ylabel('VAN por m² rotado (USD)')
ax.set_title(f'Valor actual neto de la renovación (tasa {TASA:.0%}, vida {VIDA} años)')
ax.legend(fontsize=9)
guardar(fig, 'r05_van_sensibilidad')

In [ ]:
json.dump(inferencia, open(SALIDA + 'inferencia.json','w'), indent=1, default=float, ensure_ascii=False)
print(json.dumps(inferencia, indent=1, default=float, ensure_ascii=False))

## 8. Indicadores de competitividad

Traducen la estrategia a dos magnitudes de negocio: el incremento del margen anual de
la finca y la probabilidad de un año con margen negativo.

La probabilidad se calcula con una **aproximación normal sobre el riesgo residual**,
por lo que subestima el riesgo de cola: excluye la fluctuación estacional y supone
normalidad. Se interpreta como indicador de la dirección y la magnitud relativa del
cambio, no como probabilidad absoluta.

$$P(\text{margen anual} < 0) \approx \Phi\left(-\frac{\mu' w}{\sqrt{w'\hat{\Sigma}w}}\right)$$

In [ ]:
m_act = r0 * AT
filas = {'Cartera vigente': dict(margen_m2=r0, margen_usd=m_act, incremento_pct=0.0,
                                 prob_perdida_pct=stats.norm.cdf(-r0/s0)*100)}
for k, w in {'Núcleo robusto': wN, 'Plan completo': wF}.items():
    m, s = float(mu@w), float(np.sqrt(w@S@w))
    filas[k] = dict(margen_m2=m, margen_usd=m*AT, incremento_pct=(m-r0)/r0*100,
                    prob_perdida_pct=stats.norm.cdf(-m/s)*100)
comp = pd.DataFrame(filas).T
inferencia['competitividad'] = comp.to_dict()
comp.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.3))
b = ax.bar(comp.index, comp.margen_usd/1000, color=[GR, VE, AZ], width=.55)
for r_, (k, f) in zip(b, comp.iterrows()):
    txt = f'{f.margen_usd/1000:,.0f} mil USD'.replace(',', '.')
    if f.incremento_pct > 0: txt += f'\n+{f.incremento_pct:.1f} %'
    ax.text(r_.get_x()+r_.get_width()/2, f.margen_usd/1000, txt, ha='center', va='bottom', fontsize=9.5)
ax.set_ylabel('Margen anual de la superficie optimizada (miles de USD)')
ax.set_ylim(0, comp.margen_usd.max()/1000*1.2)
ax.set_title('Margen anual de la finca por escenario, a igual riesgo')
guardar(fig, 'r08_margen_por_escenario')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.3))
b = ax.bar(comp.index, comp.prob_perdida_pct, color=[RO, VE, AZ], width=.55)
for r_, v in zip(b, comp.prob_perdida_pct):
    ax.text(r_.get_x()+r_.get_width()/2, v, f'{v:.2f} %', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Probabilidad de un año con margen negativo (%)')
ax.set_ylim(0, comp.prob_perdida_pct.max()*1.25)
ax.set_title('Exposición de la finca a un año de pérdida (indicador ilustrativo)')
guardar(fig, 'r09_probabilidad_perdida')
json.dump(inferencia, open(SALIDA + 'inferencia.json','w'), indent=1, default=float, ensure_ascii=False)

## 9. Sensibilidad a los parámetros operativos

El techo de crecimiento por variedad y el límite de rotación anual no son hiperparámetros
estadísticos sino **parámetros de negocio**: los fija la capacidad real de la finca para
conseguir planta injertada y renovar camas. Conviene por tanto medir cuánto depende el
costo de oportunidad de los valores adoptados (2× de crecimiento y 20 % de rotación).

Se recorre una rejilla de seis niveles de rotación por tres techos de crecimiento,
resolviendo la optimización completa en cada combinación.

In [ ]:
def resolver_robusto(mu, S, s_obj, ub_, tur):
    """Como resolver(), pero con techo por variedad explícito y reintentos de tolerancia."""
    cons = [{'type':'eq','fun':lambda z: z[:n].sum()-1,'jac':lambda z: np.r_[np.ones(n),np.zeros(n)]},
            {'type':'ineq','fun':lambda z: z[n:]-(z[:n]-w0),'jac':lambda z: np.hstack([-I_,I_])},
            {'type':'ineq','fun':lambda z: z[n:]+(z[:n]-w0),'jac':lambda z: np.hstack([I_,I_])},
            {'type':'ineq','fun':lambda z: 2*tur-z[n:].sum(),'jac':lambda z: np.r_[np.zeros(n),-np.ones(n)]},
            {'type':'ineq','fun':lambda z: s_obj**2-z[:n]@S@z[:n],'jac':lambda z: np.r_[-2*S@z[:n],np.zeros(n)]}]
    bnds = list(zip(np.zeros(n), ub_)) + [(0,1)]*n
    medio = np.clip((w0+ub_)/2, 0, None); medio = medio/medio.sum()
    for ftol in (1e-10, 1e-8, 1e-7):
        for z0 in (np.r_[w0, np.zeros(n)], np.r_[medio, np.zeros(n)]):
            r = minimize(lambda z: -(mu@z[:n]), z0, jac=lambda z: np.r_[-mu, np.zeros(n)],
                         method='SLSQP', bounds=bnds, constraints=cons,
                         options={'maxiter':2000,'ftol':ftol})
            if r.success:
                return r.x[:n], True
    return None, False

filas = []
for tur in [.05, .10, .15, .20, .25, .30]:
    for crece in [1.5, 2.0, 3.0]:
        ub_ = np.minimum(CAP, np.maximum(w0*crece, .005))
        w, ok = resolver_robusto(mu, S, s0, ub_, tur)
        filas.append(dict(rotacion=tur, crecimiento=crece,
                          ganancia_m2=float(mu@w - r0) if ok else np.nan,
                          ganancia_usd=float((mu@w - r0)*AT) if ok else np.nan,
                          eliminadas=int(((w0>1e-4)&(w<=1e-4)).sum()) if ok else 0,
                          convergio=ok))
SEN = pd.DataFrame(filas)
SEN.to_csv(SALIDA + 'sensibilidad_parametros.csv', index=False)
print('Combinaciones sin convergencia:', int((~SEN.convergio).sum()))
SEN.pivot(index='rotacion', columns='crecimiento', values='ganancia_usd').div(1000).round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for crece, col, ls in [(1.5, GR, '--'), (2.0, VE, '-'), (3.0, AZ, ':')]:
    t = SEN[SEN.crecimiento == crece].sort_values('rotacion')
    ax.plot(t.rotacion*100, t.ganancia_usd/1000, marker='o', color=col, ls=ls, lw=2,
            label=f'Techo de crecimiento {crece:.1f}×')
base = SEN[(SEN.rotacion == .20) & (SEN.crecimiento == 2.0)].ganancia_usd.iloc[0]
ax.scatter([20], [base/1000], s=160, marker='*', color='k', zorder=5, label='Configuración adoptada')
ax.set_xlabel('Rotación máxima anual de la superficie (%)')
ax.set_ylabel('Costo de oportunidad (miles de USD/año)')
ax.set_title('Sensibilidad del costo de oportunidad a los parámetros operativos')
ax.grid(alpha=.3); ax.legend(fontsize=9)
guardar(fig, 'r10_sensibilidad_parametros')

inferencia['sensibilidad'] = {f'rot{int(r*100)}_crec{c_:.1f}': g
                              for r, c_, g in zip(SEN.rotacion, SEN.crecimiento, SEN.ganancia_usd)}
json.dump(inferencia, open(SALIDA + 'inferencia.json','w'), indent=1, default=float, ensure_ascii=False)
print(f'Configuración adoptada (20 %, 2×): {base:,.0f} USD/año'.replace(',','.'))

La ganancia crece con ambos parámetros pero con rendimientos decrecientes. Con un techo
de crecimiento de 1,5× la curva se aplana a partir del 25 % de rotación, porque el límite
efectivo deja de ser cuánta superficie se puede renovar y pasa a ser cuánto puede crecer
cada variedad. La conclusión de fondo no depende de los valores adoptados: incluso con una
rotación del 10 % la ganancia supera los 300.000 dólares anuales.